In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import json
import os

import pandas as pd
import scanpy as sc

In [ ]:
base_dir = "/bonn-epyc/projects/spatialTCR/20240719__094819__human_kidney_7_TCR"

data_folders = ["output-XETG00088__0029040__Region_4__20240719__095642"]

data_dirs = [os.path.join(base_dir, data_folder) for data_folder in data_folders]


min_qv = 20

for data_dir in data_dirs:
    # load all the relevant data for this slide
    adata = sc.read_10x_h5(os.path.join(data_dir, "cell_feature_matrix.h5"))
    sample_str = data_dir.split("-")[-1]
    adata.obs["sample"] = sample_str
    cells = pd.read_parquet(os.path.join(data_dir, "cells.parquet")).set_index(
        "cell_id"
    )

    adata.obs.index = adata.obs.index + "|" + sample_str
    cells.index = cells.index + "|" + sample_str

    # load all other data
    transcripts = pd.read_parquet(os.path.join(data_dir, "transcripts.parquet"))
    cell_boundaries = pd.read_parquet(os.path.join(data_dir, "cell_boundaries.parquet"))
    with open(os.path.join(data_dir, "gene_panel.json")) as f:
        gene_panel = json.load(f)

    # get all gene names
    targets = gene_panel["payload"]["targets"]
    genes = [
        t["type"]["data"]["name"] for t in targets if t["type"]["descriptor"] == "gene"
    ]

    # filter transcripts
    transcripts = transcripts[transcripts.qv >= min_qv]
    transcripts = transcripts[transcripts.feature_name.isin(genes)]

    # only use assigned transcripts
    transcripts = transcripts[transcripts.cell_id != "UNASSIGNED"]

    pass

In [ ]:
adata

In [ ]:
!ls $data_dir

In [ ]:
transcripts.groupby("cell_id")["feature_name"].value_counts().unstack().fillna(0)